In [ ]:
# Colab cell 1: install requirements
!pip install -q scikit-learn pandas joblib fastapi uvicorn[standard] pyngrok streamlit matplotlib plotly requests python-multipart
# pyngrok may sometimes ask for authtoken - optional below


In [ ]:
# Colab cell 2: data load, preprocessing, train model, save model
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib
import os

DATA_PATH = "/content/Indian_Kids_Screen_Time.csv"  # your uploaded file

# Load
df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)
display(df.head())

# Basic cleaning - adapt column names if needed
# Expected columns (adapt if your file differs):
# 'Age', 'Gender', 'Primary_Device', 'Avg_Daily_Screen_Time_hr', 'Educational_Avg_Screen_Time_hr', 'Recreational_Avg_Screen_Time_hr', 'Health_Impacts', ...

# Create features
df['Edu_to_Rec_ratio'] = df.apply(
    lambda r: (r.get('Educational_Avg_Screen_Time_hr', 0) / (r.get('Recreational_Avg_Screen_Time_hr', 1)+1e-6))
    if ('Educational_Avg_Screen_Time_hr' in df.columns and 'Recreational_Avg_Screen_Time_hr' in df.columns)
    else r.get('Educational_to_Recreational_Ratio', 0),
    axis=1
)

# If Combined_Recommended_Limit exists use it, else build target heuristically:
if 'Combined_Recommended_Limit_hr' in df.columns:
    target_reg = df['Combined_Recommended_Limit_hr'].astype(float)
else:
    # Heuristic: non-exceeded users mean as target if Exceeded_Recommended_Limit available
    if 'Exceeded_Recommended_Limit' in df.columns:
        proxy = df.loc[df['Exceeded_Recommended_Limit']==False, 'Avg_Daily_Screen_Time_hr'].mean()
        df['Combined_Recommended_Limit_hr'] = np.round(proxy, 2)
        target_reg = df['Combined_Recommended_Limit_hr'].astype(float)
    else:
        # fallback heuristic: smaller limit for younger ages
        df['Combined_Recommended_Limit_hr'] = np.clip(2.0 + (df['Age']>15)*0.5, 0.5, 6.0)
        target_reg = df['Combined_Recommended_Limit_hr'].astype(float)

# Classification target: whether exceeds 4 hours
df['Exceeds_4h'] = (df['Avg_Daily_Screen_Time_hr'] > 4).astype(int)

# Features to use
features = []
for c in ['Age', 'Gender', 'Primary_Device', 'Avg_Daily_Screen_Time_hr', 'Edu_to_Rec_ratio']:
    if c in df.columns:
        features.append(c)

X = df[features].copy()
y_reg = target_reg
y_clf = df['Exceeds_4h']

# Preprocessing pipeline
numeric_features = [c for c in ['Age', 'Avg_Daily_Screen_Time_hr', 'Edu_to_Rec_ratio'] if c in X.columns]
cat_features = [c for c in ['Gender', 'Primary_Device'] if c in X.columns]

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_pipe, numeric_features), ('cat', cat_pipe, cat_features)])

# Regression model pipeline
reg_pipe = Pipeline([('pre', preprocessor),
                     ('model', RandomForestRegressor(n_estimators=200, random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
reg_pipe.fit(X_train, y_train)
print("Regression model trained")

# Classification model pipeline (for 'exceeds' prediction)
clf_pipe = Pipeline([('pre', preprocessor),
                     ('model', RandomForestClassifier(n_estimators=200, random_state=42))])
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, random_state=42)
clf_pipe.fit(X_train_c, y_train_c)
print("Classification model trained")

# Save models
os.makedirs("models", exist_ok=True)
joblib.dump(reg_pipe, "models/reco_regression.pkl")
joblib.dump(clf_pipe, "models/exceed_classifier.pkl")
print("Saved models to models/reco_regression.pkl and models/exceed_classifier.pkl")


Loaded shape: (9712, 8)


,Age,Gender,Avg_Daily_Screen_Time_hr,Primary_Device,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,Health_Impacts,Urban_or_Rural
0,14,Male,3.99,Smartphone,True,0.42,"Poor Sleep, Eye Strain",Urban
1,11,Female,4.61,Laptop,True,0.30,Poor Sleep,Urban
2,18,Female,3.73,TV,True,0.32,Poor Sleep,Urban
3,15,Female,1.21,Laptop,False,0.39,NaN,Urban
4,12,Female,5.89,Smartphone,True,0.49,"Poor Sleep, Anxiety",Urban


Regression model trained
Classification model trained
Saved models to models/reco_regression.pkl and models/exceed_classifier.pkl


In [ ]:
# Colab cell 3: write fastapi server file
%%writefile app_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

app = FastAPI(title="Screen Time Recommendation API")

# load models
reco = joblib.load("models/reco_regression.pkl")
clf = joblib.load("models/exceed_classifier.pkl")

class UserInput(BaseModel):
    Age: int
    Gender: str
    Primary_Device: str
    Avg_Daily_Screen_Time_hr: float
    Educational_Avg_Screen_Time_hr: float = None
    Recreational_Avg_Screen_Time_hr: float = None
    Edu_to_Rec_ratio: float = None

@app.post("/recommend/regression")
def recommend_regression(payload: UserInput):
    # build dataframe row consistent with training features
    row = {
        'Age': payload.Age,
        'Gender': payload.Gender,
        'Primary_Device': payload.Primary_Device,
        'Avg_Daily_Screen_Time_hr': payload.Avg_Daily_Screen_Time_hr
    }
    # compute ratio if not provided
    if payload.Edu_to_Rec_ratio:
        row['Edu_to_Rec_ratio'] = payload.Edu_to_Rec_ratio
    else:
        edu = payload.Educational_Avg_Screen_Time_hr or 0
        rec = payload.Recreational_Avg_Screen_Time_hr or max(payload.Avg_Daily_Screen_Time_hr - edu, 0.0001)
        row['Edu_to_Rec_ratio'] = edu / (rec + 1e-6)
    df = pd.DataFrame([row])
    pred = float(reco.predict(df)[0])
    return {"recommended_limit_hr": round(pred, 2)}

@app.post("/recommend/classify")
def recommend_classify(payload: UserInput):
    row = {
        'Age': payload.Age,
        'Gender': payload.Gender,
        'Primary_Device': payload.Primary_Device,
        'Avg_Daily_Screen_Time_hr': payload.Avg_Daily_Screen_Time_hr
    }
    if payload.Edu_to_Rec_ratio:
        row['Edu_to_Rec_ratio'] = payload.Edu_to_Rec_ratio
    else:
        edu = payload.Educational_Avg_Screen_Time_hr or 0
        rec = payload.Recreational_Avg_Screen_Time_hr or max(payload.Avg_Daily_Screen_Time_hr - edu, 0.0001)
        row['Edu_to_Rec_ratio'] = edu / (rec + 1e-6)
    df = pd.DataFrame([row])
    prob = clf.predict_proba(df)[0].tolist()
    pred = int(clf.predict(df)[0])
    return {"exceeds_4h": bool(pred), "probabilities": prob}


Overwriting app_api.py


In [ ]:
# Colab cell 5: write the Streamlit app
%%writefile app_streamlit.py
import streamlit as st
import pandas as pd
import requests
import os
import json
import matplotlib.pyplot as plt
import streamlit.components.v1 as components

st.set_page_config(page_title="Kids Screen Time — Reco + PowerBI", layout="wide")

st.title("📊 Screen Time Recommendation + Power BI Dashboard")

# Config: set the FastAPI base URL (ngrok URL from previous cell)
API_BASE_URL = st.sidebar.text_input("FastAPI base URL (e.g. https://xxxx.ngrok.io)", value="http://localhost:8000")

st.sidebar.header("Power BI Embed")
embed_option = st.sidebar.selectbox("Choose embed method", ["Publish to web (public)", "Power BI secure embed (token - manual steps)"])
powerbi_url = st.sidebar.text_input("Power BI iframe URL (publish-to-web or embed link)", "")

# Upload / show dataset
uploaded = st.file_uploader("Upload Indian_Kids_Screen_Time CSV (optional)", type=["csv"])
if uploaded:
    df = pd.read_csv(uploaded)
else:
    # attempt to load from runtime (if you trained earlier)
    try:
        df = pd.read_csv("/mnt/data/Indian_Kids_Screen_Time.csv")
    except Exception:
        df = None

if df is not None:
    st.subheader("Dataset preview")
    st.dataframe(df.head(200))
    # simple plot: Avg screen time by age
    if 'Age' in df.columns and 'Avg_Daily_Screen_Time_hr' in df.columns:
        fig, ax = plt.subplots(figsize=(6,3))
        grouped = df.groupby('Age')['Avg_Daily_Screen_Time_hr'].mean()
        grouped.plot(kind='bar', ax=ax)
        ax.set_ylabel("Avg Daily Screen Time (hrs)")
        ax.set_xlabel("Age")
        st.pyplot(fig)

# Recommendation form
st.subheader("Get Personalized Recommendation")
with st.form("reco_form"):
    age = st.number_input("Age", min_value=1, max_value=100, value=10)
    gender = st.selectbox("Gender", options=["Male", "Female", "Other"])
    device = st.selectbox("Primary Device", options=["Smartphone", "Laptop", "Tablet", "TV"])
    avg_time = st.number_input("Avg Daily Screen Time (hrs)", min_value=0.0, max_value=24.0, value=3.0, step=0.1)
    edu_time = st.number_input("Educational Screen Time (hrs)", min_value=0.0, max_value=24.0, value=1.0, step=0.1)
    submitted = st.form_submit_button("Get Recommendation")
if submitted:
    # prepare payload
    payload = {
        "Age": int(age),
        "Gender": gender,
        "Primary_Device": device,
        "Avg_Daily_Screen_Time_hr": float(avg_time),
        "Educational_Avg_Screen_Time_hr": float(edu_time)
    }
    # Regression call
    try:
        reg_resp = requests.post(API_BASE_URL + "/recommend/regression", json=payload, timeout=10).json()
        rec_limit = reg_resp.get("recommended_limit_hr")
    except Exception as e:
        rec_limit = None
        st.error(f"Regression API call failed: {e}")

    # Classification call
    try:
        clf_resp = requests.post(API_BASE_URL + "/recommend/classify", json=payload, timeout=10).json()
        exceeds = clf_resp.get("exceeds_4h")
        probs = clf_resp.get("probabilities")
    except Exception as e:
        exceeds = None
        probs = None
        st.error(f"Classification API call failed: {e}")

    # Show results
    st.markdown("### Recommendations")
    if rec_limit is not None:
        st.info(f"Recommended daily limit (model): **{rec_limit} hrs/day**")
        diff = round(float(avg_time) - float(rec_limit), 2)
        if diff > 0:
            st.warning(f"You are spending **{diff} hrs/day** more than the recommended limit.")
        else:
            st.success(f"You're **{abs(diff)} hrs/day** below the recommended limit. Good job!")
    if exceeds is not None:
        st.write("Model says exceeds 4h?" , "✅ Yes" if exceeds else "❌ No")
        if probs:
            st.write("Probabilities (not exceed, exceed):", probs)

# Embed Power BI
st.subheader("Power BI Dashboard")

if powerbi_url:
    # Option 1: publish-to-web public link - display via iframe
    if embed_option == "Publish to web (public)":
        st.info("Using Publish-to-web embed. Note: This makes the report public if you published it that way.")
        components.iframe(powerbi_url, height=700)
    else:
        # Option 2: secure embed requires more complex token-based JS embed (here we show instructions and a placeholder)
        st.info("Secure embed requires embed token and workspace/report id. Below is a simple placeholder.")
        st.markdown("""
        **Secure embedding steps (summary)**:
        1. Publish PBIX to Power BI Service (your workspace).
        2. Use Azure AD + Power BI REST APIs to get an embed token (requires tenant/admin setup).
        3. In the Streamlit app, use a small HTML+JS (powerbi-client) snippet to render the report with token.
        """)
        # Placeholder: if user has a public iframe url we still show it
        components.iframe(powerbi_url, height=700)
else:
    st.info("Provide a Power BI embed or publish-to-web URL in the sidebar to show the dashboard here.")


Overwriting app_streamlit.py


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import pickle
import numpy as np # Added for np.clip and other numerical operations potentially needed for feature creation

df = pd.read_csv("/content/Indian_Kids_Screen_Time.csv")

# Create features consistent with previous model training
df['Edu_to_Rec_ratio'] = df.apply(
    lambda r: (r.get('Educational_Avg_Screen_Time_hr', 0) / (r.get('Recreational_Avg_Screen_Time_hr', 1)+1e-6))
    if ('Educational_Avg_Screen_Time_hr' in df.columns and 'Recreational_Avg_Screen_Time_hr' in df.columns)
    else r.get('Educational_to_Recreational_Ratio', 0),
    axis=1
)

# Create target label
df["Exceeds_Limit"] = (df["Avg_Daily_Screen_Time_hr"] > 4).astype(int)

# ML Features - using columns known to exist and consistent with prior model
# Ensure these columns exist or handle their absence gracefully if not in the dataset
features = []
for c in ['Age', 'Gender', 'Primary_Device', 'Avg_Daily_Screen_Time_hr', 'Edu_to_Rec_ratio']:
    if c in df.columns:
        features.append(c)

X = df[features]
y = df["Exceeds_Limit"]

# Preprocessing pipeline
numeric_features = [c for c in ['Age', 'Avg_Daily_Screen_Time_hr', 'Edu_to_Rec_ratio'] if c in X.columns]
cat_features = [c for c in ['Gender', 'Primary_Device'] if c in X.columns]

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_pipe, numeric_features), ('cat', cat_pipe, cat_features)])

# Train model with preprocessing in a pipeline
model_pipeline = Pipeline([('pre', preprocessor),
                           ('model', RandomForestClassifier(n_estimators=200, random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

pickle.dump(model_pipeline, open("model.pkl", "wb"))

# Save full dataset for recommendations
df.to_csv("dataset.csv", index=False)


In [ ]:
%%writefile api_backend.py
from fastapi import FastAPI
import pandas as pd
import pickle

app = FastAPI()

model = pickle.load(open("model.pkl", "rb"))
df = pd.read_csv("dataset.csv")

def get_recommendation(age, sleep, academic, outdoor):
    rec = []
    if sleep < 8:
        rec.append("Increase sleep to at least 8 hours.")
    if outdoor < 1:
        rec.append("Encourage at least 1 hour outdoor play.")
    if academic < 60:
        rec.append("Reduce screen time to improve focus & academics.")
    if age < 10 and sleep < 9:
        rec.append("Young kids need minimum 9 hours of sleep.")
    if not rec:
        rec.append("Healthy lifestyle! Continue same behavior.")
    return rec

@app.get("/")
def home():
    return {"message": "API is working"}

@app.get("/predict")
def predict(age: int, sleep: float, academic: float, outdoor: float):
    input_data = [[age, sleep, academic, outdoor]]
    pred = model.predict(input_data)[0]
    rec = get_recommendation(age, sleep, academic, outdoor)
    return {
        "prediction": "Exceeds Limit" if pred == 1 else "Within Limit",
        "recommendation": rec
    }

@app.get("/dataset")
def dataset():
    return df.to_dict(orient="records")


Overwriting api_backend.py


In [ ]:
!pip install fastapi uvicorn nest_asyncio


In [ ]:
%%writefile api_backend.py
from fastapi import FastAPI
import pandas as pd
import pickle

app = FastAPI()

model = pickle.load(open("model.pkl", "rb"))
df = pd.read_csv("dataset.csv")

def get_recommendation(age, sleep, academic, outdoor):
    rec = []
    if sleep < 8: rec.append("Increase sleep to at least 8 hours.")
    if outdoor < 1: rec.append("Encourage at least 1 hour outdoor play.")
    if academic < 60: rec.append("Reduce screen time to boost academics.")
    if age < 10 and sleep < 9: rec.append("Kids below 10 need 9+ hrs sleep.")
    if not rec: rec.append("Healthy lifestyle. Continue same.")
    return rec

@app.get("/")
def home():
    return {"message": "API working"}

@app.get("/predict")
def predict(age: int, sleep: float, academic: float, outdoor: float):
    data = [[age, sleep, academic, outdoor]]
    pred = model.predict(data)[0]
    rec = get_recommendation(age, sleep, academic, outdoor)
    return {
        "prediction": "Exceeds Limit" if pred==1 else "Within Limit",
        "recommendations": rec
    }


Overwriting api_backend.py


In [ ]:
!npm install -g localtunnel
!lt --port 8000 &


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 22 packages in 4s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸your url is: https://mean-facts-yell.loca.lt


In [ ]:
%%writefile app_streamlit.py
import streamlit as st
import requests

API_URL = "https://icy-animals-clap.loca.lt"   # paste your lt link

st.title("📱 Kids Screen Time Recommendation System")

st.sidebar.header("Enter Details")

age = st.sidebar.number_input("Age", 5, 18, 12)
sleep = st.sidebar.number_input("Sleep Hours", 0.0, 12.0, 7.0)
academic = st.sidebar.number_input("Academic Score", 0.0, 100.0, 65.0)
outdoor = st.sidebar.number_input("Outdoor Play Hours", 0.0, 5.0, 1.0)

if st.sidebar.button("Predict"):
    params = {"age":age, "sleep":sleep, "academic":academic, "outdoor":outdoor}
    result = requests.get(f"{API_URL}/predict", params=params).json()

    st.subheader("Prediction")
    st.success(result["prediction"])

    st.subheader("Recommendations")
    for r in result["recommendations"]:
        st.write("✔", r)

st.subheader("📊 Power BI Dashboard")
st.markdown("""
<iframe width="100%" height="600" src="/content/infosys.pbix" frameborder="0" allowFullScreen="true"></iframe>
""", unsafe_allow_html=True)


In [ ]:
pip install msal fastapi uvicorn requests python-dotenv


In [ ]:
# powerbi_embed_api.py
import os
import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import requests
from msal import ConfidentialClientApplication
from dotenv import load_dotenv

load_dotenv()  # optional - for local env vars

# -------------------------
# REQUIRED ENVIRONMENT VARS
# -------------------------
TENANT_ID = os.environ.get("TENANT_ID")          # Azure tenant id
CLIENT_ID = os.environ.get("CLIENT_ID")          # Azure app client id
CLIENT_SECRET = os.environ.get("CLIENT_SECRET")  # Azure app client secret

# Power BI report identifiers
GROUP_ID = os.environ.get("PBI_GROUP_ID")        # workspace id
REPORT_ID = os.environ.get("PBI_REPORT_ID")      # report id
# optional dataset id
DATASET_ID = os.environ.get("PBI_DATASET_ID", None)

# authority and scope
AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = ["https://analysis.windows.net/powerbi/api/.default"]

# Power BI REST base
PBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"

app = FastAPI(title="PowerBI Embed Token API")

# MSAL confidential client
def get_access_token():
    app_msal = ConfidentialClientApplication(
        client_id=CLIENT_ID,
        authority=AUTHORITY,
        client_credential=CLIENT_SECRET
    )
    token_response = app_msal.acquire_token_silent(SCOPE, account=None)
    if not token_response:
        token_response = app_msal.acquire_token_for_client(scopes=SCOPE)
    if "access_token" not in token_response:
        raise Exception("Could not obtain access token: " + json.dumps(token_response, indent=2))
    return token_response["access_token"]

# Endpoint to return embed info for the report
@app.get("/embedinfo")
def get_embed_info():
    """
    Returns:
    {
      "embedUrl": "...",
      "reportId": "...",
      "datasetId": "...",
      "groupId": "...",
      "embedToken": { "token": "...", "expiration": "..." }
    }
    """
    try:
        access_token = get_access_token()
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    headers = {"Authorization": f"Bearer {access_token}"}

    # 1) Get report metadata (embedUrl)
    report_url = f"{PBI_API_BASE}/groups/{GROUP_ID}/reports/{REPORT_ID}"
    r = requests.get(report_url, headers=headers)
    if r.status_code != 200:
        raise HTTPException(status_code=r.status_code, detail=f"Error fetching report: {r.text}")
    report_data = r.json()
    embed_url = report_data.get("embedUrl")
    dataset_id = report_data.get("datasetId", DATASET_ID)

    # 2) Generate embed token POST /groups/{groupId}/reports/{reportId}/GenerateToken
    gen_token_url = f"{PBI_API_BASE}/groups/{GROUP_ID}/reports/{REPORT_ID}/GenerateToken"
    # Body: desired access level and optional identities for RLS
    body = {
        "accessLevel": "View"  # or "Edit" if needed
    }
    token_resp = requests.post(gen_token_url, headers={**headers, "Content-Type":"application/json"}, json=body)
    if token_resp.status_code != 200:
        raise HTTPException(status_code=token_resp.status_code, detail=f"Error generating embed token: {token_resp.text}")
    token_json = token_resp.json()
    embed_token = token_json.get("token")
    expiration = token_json.get("expiration")

    return {
        "embedUrl": embed_url,
        "reportId": REPORT_ID,
        "datasetId": dataset_id,
        "groupId": GROUP_ID,
        "embedToken": {"token": embed_token, "expiration": expiration}
    }


In [ ]:
{
  "accessLevel": "View",
  "identities": [
    {
      "username": "user@domain.com",
      "roles": ["SalesRegionRole"],
      "datasets": ["<datasetId>"]
    }
  ]
}


In [ ]:
!pip install streamlit


In [ ]:
!streamlit run app_streamlit.py --server.port 8501 --server.address 0.0.0.0 &


In [ ]:
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate


In [ ]:
# app_streamlit.py
import streamlit as st
import requests
import streamlit.components.v1 as components
import os

st.set_page_config(page_title="Secure Power BI Embed", layout="wide")
st.title("🔐 Secure Power BI Embed — Demo")

# Backend endpoint that returns embed info
EMBED_API = st.sidebar.text_input("Embed API endpoint", value="http://localhost:8001/embedinfo")

if st.sidebar.button("Get embed info"):
    try:
        resp = requests.get(EMBED_API, timeout=10)
        resp.raise_for_status()
        info = resp.json()
    except Exception as e:
        st.error(f"Failed to get embed info: {e}")
        st.stop()

    embed_url = info["embedUrl"]
    embed_token = info["embedToken"]["token"]
    report_id = info["reportId"]
    group_id = info["groupId"]

    st.write("Embed URL:", embed_url)
    st.write("Report ID:", report_id)
    st.write("Group ID:", group_id)

    # Build the HTML + JS to render the report using powerbi-client
    # We provide a small script that uses embed token and embed URL to render the report
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
      <meta charset="utf-8" />
      <script src="https://cdn.jsdelivr.net/npm/powerbi-client@2.19.0/dist/powerbi-client.min.js"></script>
      <style>
        body, html, #reportContainer {{ height: 100%; margin:0; padding:0; }}
        #reportContainer {{ height: 800px; }}
      </style>
    </head>
    <body>
      <div id="reportContainer"></div>
      <script>
        (function() {{
          const models = window['powerbi-client'].models;
          const embedConfiguration = {{
            type: 'report',
            tokenType: models.TokenType.Embed,
            accessToken: "{embed_token}",
            embedUrl: "{embed_url}",
            id: "{report_id}",
            settings: {{
              panes: {{
                filters: {{ visible: false }},
                pageNavigation: {{ visible: true }}
              }},
              navContentPaneEnabled: true
            }}
          }};
          const reportContainer = document.getElementById('reportContainer');
          // reset
          window.powerbi.reset(reportContainer);
          // embed
          const report = window.powerbi.embed(reportContainer, embedConfiguration);

          // optional: handle loaded / error
          report.on('loaded', function() {{
            console.log('Report loaded');
          }});
          report.on('error', function(event) {{
            console.error(event.detail);
            const err = document.createElement('div');
            err.innerHTML = '<pre>' + JSON.stringify(event.detail, null, 2) + '</pre>';
            document.body.appendChild(err);
          }});
        }})();
      </script>
    </body>
    </html>
    """

    components.html(html, height=820, scrolling=True)
else:
    st.info("Click 'Get embed info' to fetch an embed token and render the secure Power BI report.")
